# FinChart-R2 - Phase 2C: Multimodal DPO on SFT-408

**Colab-only diagnostic notebook.** It continues the public SFT adapter Kxck/Finance_500_v1 with 51 structured visual preference pairs using TRL's VLM-supported DPO path.

Research boundary: all 51 pairs were derived from frozen ChartQA val[0:500]. This run may validate the DPO path and inspect training metrics, but it must not be evaluated on or compared with the frozen 69.0% SFT-408 benchmark. A reportable experiment requires equivalent pairs derived solely from ChartQA train.


## 1. Colab runtime

In Colab select **Runtime -> Change runtime type -> T4 GPU** (or stronger) and Runtime Version **2026.07**. This stable runtime uses Python 3.12.13 and NumPy 2.0.2; do not use the newer Python 3.13 runtime for this notebook.


In [10]:
# !pip install -q --no-cache-dir --upgrade "numpy==2.0.2" "pillow==11.3.0"
# !pip install -q -U --no-cache-dir unsloth unsloth_zoo transformers accelerate bitsandbytes peft datasets
# !pip install -q --no-cache-dir --upgrade --force-reinstall "trl==1.9.2"
!pip install -q --no-cache-dir --upgrade "torchvision>=0.28.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 254.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.8.19 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.1 which is incompatible.
unsloth 2026.8.19 requires torch<2.12.0,>=2.4.0, but you have torch 2.13.0 which is incompatible.
unsloth 2026.8.19 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.15.1 which is incompatible.
unsloth 2026.8.19 requires trl!=0.19.0,<=0.24.0,>=0.18.2, but you have trl 1.9.2 which is incompatible.


### Restart required after installation

After the installation cell finishes, select **Runtime -> Restart session**, then resume at Mount Google Drive. This loads the pinned TRL VLM-DPO API and prevents mixed NumPy/Pillow binaries after package installation.


## 2. Mount Google Drive

Copy phase2c_orpo_51_pairs.jsonl into MyDrive/FinChart-R2/data/. Despite the historical filename, its schema is generic preference data (prompt, chosen, rejected) and is used directly by DPO.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

ROOT = Path('/content/drive/MyDrive')
PROJECT_DIR = ROOT / 'FinChart-R2'
DATA_DIR = PROJECT_DIR / 'data'
RUN_DIR = PROJECT_DIR / 'phase2c' / 'dpo_51_leaky_diagnostic'
ADAPTER_DIR = RUN_DIR / 'adapter_sft_dpo_51'
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
for path in [RUN_DIR, ADAPTER_DIR, CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

PAIR_SOURCE = DATA_DIR / 'phase2c_orpo_51_pairs.jsonl'
SFT_ADAPTER_ID = 'Kxck/Finance_500_v1'
BASE_MODEL = 'unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit'
print('Pair source:', PAIR_SOURCE)
print('SFT adapter:', SFT_ADAPTER_ID)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pair source: /content/drive/MyDrive/FinChart-R2/data/phase2c_orpo_51_pairs.jsonl
SFT adapter: Kxck/Finance_500_v1


## 3. Load pairs and enforce the research boundary


In [3]:
import json

def read_jsonl(path):
    with path.open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

if not PAIR_SOURCE.exists():
    raise FileNotFoundError(f'Upload the pair artifact first: {PAIR_SOURCE}')

records = read_jsonl(PAIR_SOURCE)
assert len(records) == 51, f'Expected 51 pairs, got {len(records)}'
required = {'prompt', 'chosen', 'rejected', 'image_split', 'image_index', 'dataset_index'}
for row in records:
    assert required <= row.keys(), f'Missing required fields: {required - row.keys()}'
    assert row['chosen'].strip() != row['rejected'].strip(), 'Degenerate preference pair.'

split_counts = {}
for row in records:
    split_counts[row['image_split']] = split_counts.get(row['image_split'], 0) + 1
uses_frozen_validation = any(
    row['image_split'] == 'val' and int(row['image_index']) < 500 for row in records
)
assert uses_frozen_validation, 'This notebook is intentionally scoped to the current validation-derived diagnostic.'
print('Pair count:', len(records))
print('Pair splits:', split_counts)
print('LEAKY DIAGNOSTIC: frozen evaluation is disabled for this adapter.')


Pair count: 51
Pair splits: {'val': 51}
LEAKY DIAGNOSTIC: frozen evaluation is disabled for this adapter.


## 4. Build a VLM preference dataset

TRL DPO receives the chart as a top-level images column plus conversational prompt, chosen, and rejected columns. The user prompt carries text only; the DPO vision collator injects the image placeholder from the images column. Do not use UnslothVisionDataCollator here.


In [4]:
from datasets import Dataset, load_dataset

DATASET_NAME = 'HuggingFaceM4/ChartQA'
datasets_by_split = {
    split: load_dataset(DATASET_NAME, split=split)
    for split in sorted({row['image_split'] for row in records})
}

examples = []
for row in records:
    image = datasets_by_split[row['image_split']][int(row['image_index'])]['image']
    examples.append({
        'prompt': [{'role': 'user', 'content': row['prompt']}],
        'chosen': [{'role': 'assistant', 'content': row['chosen']}],
        'rejected': [{'role': 'assistant', 'content': row['rejected']}],
        'images': [image],
        'dataset_index': int(row['dataset_index']),
    })

dpo_dataset = Dataset.from_list(examples)
assert len(dpo_dataset) == 51
assert 'images' in dpo_dataset.column_names
print(dpo_dataset)
print('Chosen example:\n', records[0]['chosen'])
print('Rejected example:\n', records[0]['rejected'])


README.md:   0%|          | 0.00/852 [00:00<?, ?B/s]

data/train-00000-of-00003-49492f364babfa(…): reconstructing file:   0%|          |  0.00B /  219MB            

data/train-00000-of-00003-49492f364babfa(…): downloading bytes:           |  0.00B            

data/train-00001-of-00003-7302bae5e425bb(…): reconstructing file:   0%|          |  0.00B /  311MB            

data/train-00001-of-00003-7302bae5e425bb(…): downloading bytes:           |  0.00B            

...

data/test-00000-of-00001-e2cd0b7a0f9eb20(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/28299 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1920 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected', 'images', 'dataset_index'],
    num_rows: 51
})
Chosen example:
 Target series: Mostly bad news
Target color: red
Relevant value(s): 61
Visual diagnosis: WRONG_VALUE
Answer: 37
Rejected example:
 Answer: 43


In [8]:
!pip install --upgrade "torchvision>=0.28.0

/bin/bash: -c: line 1: unexpected EOF while looking for matching `"'
/bin/bash: -c: line 2: syntax error: unexpected end of file


## 5. Load the SFT-408 policy adapter


In [11]:
import unsloth
import torch
from peft import PeftModel
from unsloth import FastVisionModel, is_bfloat16_supported

if not torch.cuda.is_available():
    raise RuntimeError('Start a Colab GPU runtime before continuing.')

model, processor = FastVisionModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
)
model = PeftModel.from_pretrained(model, SFT_ADAPTER_ID, is_trainable=True)
processor.tokenizer.padding_side = 'left'
FastVisionModel.for_training(model)
print('Loaded trainable SFT-408 policy adapter.')


/usr/local/lib/python3.13/dist-packages/unsloth/_gpu_init.py:98: UserWarning: Unsloth: torchaudio cannot initialise against this torch and has been disabled for this process, so anything that needs it will report it as missing rather than crash at import. Install the matching wheel to restore it. Original error: Detected that PyTorch and TorchAudio were compiled with different CUDA versions. PyTorch has CUDA version 13.0 whereas TorchAudio has CUDA version 12.8. Please install the TorchAudio version that matches your PyTorch version.
  disable_torchaudio_if_cuda_mismatched()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0821 20:29:09.209000 10000 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0821 20:29:09.312000 10000 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:1989: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


...

[ERROR] `min_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
[ERROR] `max_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.


adapter_config.json:   0%|          | 0.00/1.59k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  157MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Loaded trainable SFT-408 policy adapter.


## 6. DPO VLM preflight

TRL DPO supports a top-level images column for Vision-Language Models. max_length=None is required here so image tokens are not truncated. The explicit TRL vision-preference collator removes ambiguity from version-specific automatic collator selection. The batch must contain pixel_values before training.


In [12]:
import trl
from packaging.version import Version
assert Version(trl.__version__) >= Version('1.9.2'), (
    f'TRL {trl.__version__} is too old for this VLM DPO notebook. '
    'Rerun Cell 1, restart the runtime, then start again from Cell 2.'
)

from trl import DPOConfig, DPOTrainer
from trl.trainer.dpo_trainer import DataCollatorForVisionPreference

DPO_BETA = 0.05
dpo_args = DPOConfig(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=5e-6,
    warmup_steps=5,
    beta=DPO_BETA,
    max_length=None,
    remove_unused_columns=False,
    logging_steps=1,
    save_strategy='epoch',
    optim='adamw_8bit',
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    report_to='none',
)

# Explicit VLM collator: do not replace with UnslothVisionDataCollator.
vision_collator = DataCollatorForVisionPreference(
    processor=processor,
    max_length=None,
)

# ref_model=None makes DPO retain the initial policy state as its reference.
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=processor,
    data_collator=vision_collator,
)

batch = next(iter(trainer.get_train_dataloader()))
print('DPO collator:', type(trainer.data_collator).__name__)
print('Batch keys:', sorted(batch.keys()))
assert 'pixel_values' in batch, 'DPO VLM preflight failed: image tensors are absent.'
print({key: tuple(value.shape) for key, value in batch.items() if hasattr(value, 'shape')})
print('DPO VLM preflight passed.')


DPO collator: DataCollatorForVisionPreference
Batch keys: ['attention_mask', 'completion_mask', 'image_grid_thw', 'input_ids', 'mm_token_type_ids', 'pixel_values']
{'input_ids': (2, 594), 'attention_mask': (2, 594), 'mm_token_type_ids': (2, 594), 'pixel_values': (4104, 1536), 'image_grid_thw': (2, 3), 'completion_mask': (2, 594)}
DPO VLM preflight passed.


## 7. One-epoch DPO diagnostic

This trains and saves an SFT + DPO diagnostic adapter. Track rewards/accuracies and rewards/margins; positive and improving values show the preference objective is separating chosen from rejected completions. They do not replace task accuracy.


In [13]:
trainer_stats = trainer.train()
model.save_pretrained(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))

metadata = {
    'source_adapter': SFT_ADAPTER_ID,
    'pair_source': str(PAIR_SOURCE),
    'pair_count': len(records),
    'method': 'DPO',
    'beta': DPO_BETA,
    'epochs': 1,
    'training_data_boundary': 'VALIDATION_DERIVED_LEAKY_DIAGNOSTIC_ONLY',
    'frozen_evaluation_permitted': False,
}
(RUN_DIR / 'dpo_run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved SFT + DPO adapter:', ADAPTER_DIR)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51 | Num Epochs = 1 | Total steps = 7
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 39,321,600 of 4,516,459,008 (0.87% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logits / chosen,logits / rejected,logps / chosen,logps / rejected
1,0.693587,-0.000832,0.000047,0.375000,-0.000880,3.913631,3.986102,-106.168642,-10.747218
2,0.692588,0.000735,-0.000385,1.000000,0.001120,3.977820,3.989829,-99.269176,-12.958553
3,0.680498,0.024312,-0.001155,1.000000,0.025467,3.972383,3.955022,-106.644548,-18.203275
4,0.659454,0.065292,-0.003328,1.000000,0.068620,3.990399,3.876859,-95.133668,-11.994527
5,0.629677,0.123845,-0.007982,1.000000,0.131827,4.235702,3.941856,-102.186331,-7.699842
6,0.595418,0.194548,-0.012425,1.000000,0.206973,4.070872,4.122057,-90.453485,-7.007611
7,0.521348,0.358648,-0.022315,1.000000,0.380963,3.357648,4.314863,-97.536082,-19.801582


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FinChart-R2/phase2c/dpo_51_leaky_diagnostic/checkpoints/checkpoint-7/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FinChart-R2/phase2c/dpo_51_leaky_diagnostic/adapter_sft_dpo_51/tokenizer_config.json.


Saved SFT + DPO adapter: /content/drive/MyDrive/FinChart-R2/phase2c/dpo_51_leaky_diagnostic/adapter_sft_dpo_51


## 8. Frozen evaluation is intentionally blocked

Do not use ChartQA val[0:500] after this run. Rebuild preference pairs from ChartQA train, rerun the same notebook with uses_frozen_validation == False, then reuse the unchanged Phase 1 evaluator from notebook 03.


In [ ]:
raise RuntimeError(
    'Frozen evaluation blocked: this DPO adapter was trained on preference pairs derived from val[0:500]. '
    'Its score cannot be compared to the reported 69.0% SFT-408 frozen benchmark.'
)
